# Remote Pyserini Query

Minimal notebook for querying a hosted Pyserini API configured in `.env`.

It loads `.env` from this worktree or the shared checkout, reads a typed config, sends query requests, and prints the top returned documents. It does not parse topics, open Lucene indexes, or download anything.


## ClimbMix Docids

A returned docid such as `shard_04149_6425` is the official ClimbMix document id. Read it as row `6425` from source Parquet shard `shard_04149.parquet`. Treat it as the stable id to fetch, cite, or submit exactly as returned; it is not a rank, score, topic id, or semantic category.

## Default Query

By default this notebook uses a real local development-set query: `rag25-topics-dev.tsv` topic `31`, about e-waste, waste management, recycling, sustainability, and local economies. Set `SAMPLE_QUERIES` to override it with semicolon-separated queries.

In [1]:
from __future__ import annotations

import sys
import textwrap
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "code"))

from trec_rag.remote_pyserini import (  # noqa: E402
    RemotePyseriniClient,
    RemotePyseriniConfig,
    load_repo_env,
)

load_repo_env(REPO_ROOT)
CONFIG = RemotePyseriniConfig.from_env()
CLIENT = RemotePyseriniClient(CONFIG)

print(f"Repo root: {REPO_ROOT}")
print(f"Index URL: {CONFIG.index_url}")
print(f"Token present: {bool(CONFIG.api_token)}")
print(f"Hits per query: {CONFIG.hits}")
print(f"Queries: {len(CONFIG.queries)}")

Repo root: /home/nidhin/.codex/worktrees/f260/trec_rag
Index URL: http://api.castorini.uwaterloo.ca/v1/climbmix-400b/search
Token present: True
Hits per query: 5
Queries: 1


In [2]:
import pandas as pd
from IPython.display import display

all_results = []
for query in CONFIG.queries:
    rows = CLIENT.candidates(query)
    all_results.append({"query": query, "candidates": rows})

    print("Query:")
    print(query)
    print(f"Returned {len(rows)} candidates.")

    preview_rows = [
        {
            "rank": row["rank"],
            "score": row["score"],
            "docid": row["docid"],
            "length": row["text_length"],
            "preview": textwrap.shorten(str(row["text"]), width=220, placeholder=" ..."),
        }
        for row in rows
    ]
    display(pd.DataFrame(preview_rows))

    if rows:
        top = rows[0]
        print(f"Full content for top document {top['docid']}")
        print(f"Score: {top['score']}  Length: {top['text_length']:,} characters")
        print(str(top["text"]))


Query:
I'm seeking to understand the environmental and health impacts of e-waste and other waste, along with how proper recycling benefits sustainability and local economies. Could you explain the risks of improper waste management, recent recycling innovations, and practical steps individuals and businesses can take for responsible waste handling?
Returned 5 candidates.


,rank,score,docid,length,preview
0,1,64.097000,shard_03212_30900,14366,The Importance of Laptop Recycling in Corporat...
1,2,63.968102,shard_00230_9843,13699,"In today's digital age, technology has become ..."
2,3,63.687302,shard_03861_56160,31467,Waste Management in the Environment: Informati...
3,4,63.283199,shard_06355_16088,23774,Search This Blog The Global Impact of E-Waste ...
4,5,63.105598,shard_04358_53317,23596,Navigating Electronic Waste Recycling: Best Pr...


Full content for top document shard_03212_30900
Score: 64.097  Length: 14,366 characters
The Importance of Laptop Recycling in Corporate E-Waste Management Proper e-waste management is crucial for businesses in the UK to address the growing concern of electronic waste. The average UK household owns nearly 25 electronic devices, and businesses require even more IT equipment. However, only 20% of annual e-waste in the UK is properly recycled, highlighting the need for sustainable e-waste management practices. Recycling not only protects the environment but also creates economic opportunities and reduces health risks associated with improper disposal. Compliance with legal regulations, such as the WEEE standard, is essential for businesses to ensure responsible e-waste management. By following the principles of reduce, reuse, and recycle, businesses can contribute to a healthier society and a greener future. Partnering with professional recycling companies and implementing green IT initia

In [3]:
total_candidates = sum(len(result["candidates"]) for result in all_results)
print(f"Remote Pyserini returned {total_candidates} candidates across {len(all_results)} queries.")

Remote Pyserini returned 5 candidates across 1 queries.


## Run

```bash
uv run --with jupyter --with nbconvert --with ipykernel --with pandas python -m jupyter nbconvert --execute --to notebook --inplace reports/notebooks/pyserini_collection_exploration.ipynb
```

Endpoint env: set `INDEX_URL`; or set `PYSERINI_INDEX` and `PYSERINI_BASE_URL`; or use the aliases `DEFAULT_INDEX` and `DEFAULT_BASE_URL`.

Other useful env vars: `PYSERINI_API_TOKEN`, `EXTERNAL_PYSERINI_HITS`, and `SAMPLE_QUERIES` separated by semicolons.
